In [ ]:
!pip install pyspark==4.0.1 grpcio grpcio-status googleapis-common-protos pyarrow

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .remote("sc://spark-connect.kubeflow-user-example-com.svc.cluster.local:15002") \
    .getOrCreate()

# Test it
df = spark.range(100)
df.show()
print("Spark version:", spark.version)

In [ ]:
from pyspark.sql.functions import col, rand, when

df = spark.range(1000) \
    .withColumn("value", rand()) \
    .withColumn("category", when(col("id") % 3 == 0, "A")
                            .when(col("id") % 3 == 1, "B")
                            .otherwise("C"))

df.groupBy("category") \
  .agg({"value": "avg", "id": "count"}) \
  .orderBy("category") \
  .show()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col
import time

spark = SparkSession.builder \
    .remote("sc://spark-connect.kubeflow-user-example-com.svc.cluster.local:15002") \
    .getOrCreate()

# Generate large dataset and do heavy aggregations to keep executors busy
df = spark.range(0, 100_000_000, numPartitions=100) \
    .withColumn("rand1", rand()) \
    .withColumn("rand2", rand()) \
    .withColumn("rand3", rand()) \
    .withColumn("bucket", (col("id") % 1000).cast("string"))

# Force multiple shuffle stages to keep it running ~1 min
result = df.groupBy("bucket") \
    .agg({"rand1": "sum", "rand2": "avg", "rand3": "max"}) \
    .orderBy("bucket") \
    .count()

print(f"Done! Processed {result} buckets")